# Машинное обучение. ВМК МГУ

# Практическое задание 11: Градиентный бустинг деревьев - часть 2

## Уровень: **Базовый (Base)**

# О формате сдачи

🔷 При решении ноутбука используйте данный шаблон

```
✅ Можно добавлять новые ячейки любых типов
❌ Не нужно удалять текстовые ячейки c разметкой частей ноутбука и формулировками заданий
```

🔷 При оценивании задач учитывается код

```
✅ Задания, в которых необходим код, обычно помечаются фразами "Your code here"/"Ваш код" и аналогичными
❌ Ответы на вопросы без сопутствующего кода оцениваются в 0 баллов
❌ Наличе работоспособного кода в ноутбуке, если на сказано иного, обязательно
```

🔷 При оценивании задач учитываются выводы

```
✅ Задания, в которых необходимы выводы, обычно помечаются фразами Вывод"/"Ответ на вопрос"/"Ваш текст" и аналогичными
✅ Обычно выводы подразумевают под собой текстовый ответ (можно писать markdown, latex).
✅ Сопутствующие изображения, графики, таблички - приветствуются!
❌ При отсутствии выводов задание не засчитается на полный балл
```

В этом задании вы..:

- Узнаете, настройка каких гиперпараметров в бустингах приведет к успеху
- Изучите дополнительную библиотеку нахождения наилучших гиперпараметров
- Примените все полученные знания для получения лучшего скора на датасете фильмов

**Примерное время выполнения (execution
time/время выполнения, если нажать run all) всех ячеек ноутбука при
правильной реализации: 60 минут **

# Подготовка рабочей среды

Сначала установим нужные нам версии библиотек. Мы гарантируем, что в данных версиях задание будет корректно отрабатывать.

После установки нужных версий, возможно, нужно перезагрузить среду (runtime), но скорее всего вам это не понадобится

На скачивание файла и установку понадобится не более 5 минут.

**Важно!**

Устанавливать нужные версии нужно каждый раз, когда создается новый
рантайм. Например, если вы 2 часа подряд делаете это задание, то
подготовить библиотеки достаточно 1 раз. Но если вы, например, начали в
понедельник, затем закрыли/выключили ноутбук, то при продолжении в
среду, вам нужно будет запустить рантайм заново и следовательно заново
установить библиотеки.

**Важно!**
Если вы предпочитаете делать практические задания на своем личном ноутбуке, то проверьте, что вы установили рабочее окружение в соответствии с гайдом .pdf)

In [ ]:
# !!! Данный блок будет работать только в Google-Colab !!!
! gdown 10k8Hwn9kpK9SpK4IEj4-EaWQZqgYT5-Q
! pip install -r /content/requirements_2024_25_for_colab_small.txt

Downloading...
From: https://drive.google.com/uc?id=10k8Hwn9kpK9SpK4IEj4-EaWQZqgYT5-Q
To: /content/requirements_2024_25_for_colab_small.txt
100% 375/375 [00:00<00:00, 1.27MB/s]


In [ ]:
import catboost
assert(catboost.__version__ == '1.2.7')

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
<ipython-input-8-5a9f2c593962> in <cell line: 0>()
----> 1 import catboost
      2 assert(catboost.__version__ == '1.2.7')

/usr/local/lib/python3.11/dist-packages/catboost/__init__.py in <module>
----> 1 from .core import (
      2     FeaturesData, EFstrType, EShapCalcType, EFeaturesSelectionAlgorithm, EFeaturesSelectionGrouping,
      3     Pool, CatBoost, CatBoostClassifier, CatBoostRegressor, CatBoostRanker, CatBoostError, cv, sample_gaussian_process, train,
      4     sum_models, _have_equal_features, to_regressor, to_classifier, to_ranker, MultiRegressionCustomMetric,
      5     MultiRegressionCustomObjective, MultiTargetCustomMetric, MultiTargetCustomObjective

/usr/local/lib/python3.11/dist-packages/catboost/core.py in <module>
     32 
     33 try:
---> 34     from pandas import DataFrame, Series
     35 except ImportError:


Теперь можно приступать к выполнению задания! :)

## Введение

Привет, ребятушки!

Сегодня мы с вами продолжим изучать градиентный бустинг, а точнее -
научимся выбивать из него наилучшее качество! Также решим задачку на
фильмах с помощью полученных знаний.

**Внимание!** Во всех заданиях в качестве целевой метрики используется MAE (средняя абсолютная ошибка).

Значение MAE вычисляется как

M A E = N ∑ i = 1 | a ( x i ) − y i | N , M A E = ∑ i = 1 N | a ( x i ) − y i | N ,

где N N - число объектов в тестовой выборке, x i x i - вектор признаков i-го объекта, a ( x i ) a ( x i ) - предсказание на i-ом объекте, y i y i - значение целевого признака на i-м объекте.

## Установка дополнительных библиотек.

В этом задании нам понадобятся три бибиотеки, которые вы изучили в прошлом задании. Напомним документацию:

**XGBoost**: Документация здесь .
**LightGBM**: Документация здесь . Также дополнительно про установку тут .
**Catboost**: Документация здесь . Можно найти также некоторую информацию на русском тут .
**HyperOpt**: Документация здесь .

**Внимание!** Вникать и подробно
читать документацию к каждой библиотеке нет необходимости! Достаточно
обращаться туда для нахождения примеров обучения.

## Подготовка датасета

Работать будем с тем же датасетом, что и в прошлом задании

При работе в google colab для скачивания датасета достаточно запустить следующую ячейку.

При работе с ноутбуком на локальном компьютере Вы можете скачать файл по этой ссылке и чуть ниже заменить /content/dataframe_YesIndex_YesHeader_C.csv (в строке с read_csv ) на ваш локальный путь до файла.

In [ ]:
# при локальном выполнении запускать эту ячейку НЕ НАДО
!gdown 1gdDv2kTCEkF3ia1vvbvRFJM0YfqmPplb

Downloading...
From: https://drive.google.com/uc?id=1gdDv2kTCEkF3ia1vvbvRFJM0YfqmPplb
To: /content/dataframe_YesIndex_YesHeader_C.csv
100% 568k/568k [00:00<00:00, 79.7MB/s]


In [ ]:
%matplotlib inline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer

from hyperopt import hp, tpe, Trials
from hyperopt.fmin import fmin
from hyperopt.pyll import scope

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

import matplotlib.pyplot as plt

import pandas as pd

import numpy as np

import time

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
<ipython-input-7-222f200b6fcf> in <cell line: 0>()
      1 get_ipython().run_line_magic('matplotlib', 'inline')
----> 2 from sklearn.ensemble import GradientBoostingRegressor
      3 from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
      4 from sklearn.metrics import mean_absolute_error, make_scorer
      5 

/usr/local/lib/python3.11/dist-packages/sklearn/__init__.py in <module>
     81         _distributor_init,  # noqa: F401
     82     )
---> 83     from .base import clone
     84     from .utils._show_versions import show_versions
     85 

/usr/local/lib/python3.11/dist-packages/sklearn/base.py in <module>
     17 from ._config import config_context, get_config
     18 from .exceptions import InconsistentVersionWarning
---> 19 from .utils import _IS_32BIT
     20 from .utils._estimator_html_repr 

In [ ]:
df = pd.read_csv('/content/dataframe_YesIndex_YesHeader_C.csv', index_col=0)
df.head()

,Engine Capacity,Cylinders,Drive Type,Fuel Tank Capacity,Fuel Economy,Fuel Type,Horsepower,Torque,Transmission,Top Speed,...,Acceleration,Length,Width,Height,Wheelbase,Trunk Capacity,name,price,currency,Country
0,1.2,3,0,42.0,4.9,0,76,100.0,0,170,...,14.0,4.245,1.670,1.515,2.550,450.0,Mitsubishi Attrage 2021 1.2 GLX (Base),34099.0,0,0
1,1.2,3,0,42.0,4.9,0,76,100.0,0,170,...,14.0,4.245,1.670,1.515,2.550,450.0,Mitsubishi Attrage 2021 1.2 GLX (Base),34099.0,0,0
2,1.4,4,0,45.0,6.3,0,75,118.0,1,156,...,16.0,3.864,1.716,1.721,2.513,2800.0,Fiat Fiorino 2021 1.4L Standard,41250.0,0,0
3,1.6,4,0,50.0,6.4,0,102,145.0,0,180,...,11.0,4.354,1.994,1.529,2.635,510.0,Renault Symbol 2021 1.6L PE,44930.0,0,0
4,1.5,4,0,48.0,5.8,0,112,150.0,0,170,...,10.9,4.314,1.809,1.624,2.585,448.0,MG ZS 2021 1.5L STD,57787.0,0,0


Также, как и в прошлом задании, будем использовать дефолтные параметры для некоторых экспериментов:

In [ ]:
test_parameters = {"n_estimators": 1000, "max_depth": 5, "learning_rate":0.1}

### **Задание 0 [0 баллов]**

Здесь нужно по аналогии с предыдущим заданием создать 3 датасета, а
потом их разбить на тренировочное и тестовое множества. Дублируем ниже
условие

Данные : датасет со стоимостью подержанных автомобилей
Цели : В данном задании следует выполнить следующие пункты:

1. Изучить датасет, проверить наличие пропусков. Под пропусками подразумевается значение N/A вместо признака. При необходимости заменить их на среднее значение признака.
2. Добавить столбец brand с информацией о производителе автомобиля (для простоты можно взять первое слово в названии модели).
3. Решить, какие признаки Вы считаете категориальными. Конвертировать выбранные категориальные столбцы в тип category.
4. Создать датасет А , в котором выбранные категориальные
признаки установлены как категориальные. Для этого необходимо создать
вектор целевых значений Y (столбец цен автомобилей) и матрицу признаков
X, в которой все категориальные переменные помечены как
.astype('category'). Дополнительно стоит создать список с названиями и
индексами столбцов категориальных переменных (поможет в будущем).
5. Создать датасет B без категориальных признаков. Для этого необходимо просто удалить из матрицы признаков все категориальные переменные.
6. Создать датасет C , в котором выбранные категориальные
признаки закодированы через one-hot encoding. Для этого необходимо из
матрицы признаков удалить выбранные категориальные переменные, а затем
добавить новые признаки, соответствующие one-hot encoding этих
категориальных переменных (со всей этой магией поможет простая функция pd.get_dummies ).
7. Разбить датасеты на тренировочное и тестовое множества , используя train_test_split(X, y, test_size=0.25, random_state=0) (зафиксировав random_seed мы получим одинаковое разбиение на обучение/тест для всех трёх выборок).

In [ ]:
if df.isna().sum().sum() == 0: print("Пропусков нет")
df['brand'] = df['name'].str.split(' ').str[0]
df = df.drop(columns = ['name'])
df.columns

Пропусков нет


Index(['Engine Capacity', 'Cylinders', 'Drive Type', 'Fuel Tank Capacity',
       'Fuel Economy', 'Fuel Type', 'Horsepower', 'Torque', 'Transmission',
       'Top Speed', 'Seating Capacity', 'Acceleration', 'Length', 'Width',
       'Height', 'Wheelbase', 'Trunk Capacity', 'price', 'currency', 'Country',
       'brand'],
      dtype='object')


In [ ]:
datasets = {'A' : None, 'B': None, 'C': None}

category_types = ["brand", "Fuel Type", "Seating Capacity", "Country", "Drive Type", "currency"]
df[category_types] =  df[category_types].astype("category")


dfA= df.copy()
dfY = dfA['price']
dfA = dfA.drop(columns=['price'])
dfB = dfA.drop(columns=category_types)
dfB = dfA.drop(columns=category_types).copy()
dfC = pd.get_dummies(dfA, columns=category_types)

datasets['A'] = train_test_split(dfA, dfY, test_size=0.25, random_state=0)
datasets['B'] = train_test_split(dfB, dfY, test_size=0.25, random_state=0)
datasets['C'] = train_test_split(dfC, dfY, test_size=0.25, random_state=0)

**Внимание!** Некоторые библиотеки
требует дополнительной предобработки данных перед использованием моделей
из них. Подробнее вы можете найти информацию об этом в соответствующих
заданиях прошлого ноутбука (часть 1)

## Оптимизация параметров

Итак,
в прошлом задании мы с вами научились пользоваться библиотеками для
градиентного бустинга. Пришло время заняться самой интересной (нет)
частью исследований, а именно подбором параметров!

## Как правильно перебирать параметры

В
этом ноутбуке мы будем несколько раз заниматься поиском оптимальных
параметров для градиентного бустинга, перебирая задания по заданной
сетке. В этом задании от Вас не будет требоваться найти самые лучшие
параметры, но всё равно важно правильно составлять сетку для перебора.
Для этого нужно понимать суть параметров и их смысл.

При настройке моделей градиентного бустинга важно правильно подбирать
гиперпараметры, так как они значительно влияют на качество
предсказаний. Большая часть параметров так или иначе пересекается, но у
каждой библиотеки они могут называться по-своему.

Ниже приведена таблица основных параметров, которые стоит перебирать для каждой из библиотек.

## 🚀 XGBoost (XGBClassifier, XGBRegressor)

Параметр
Описание
Диапазон перебора
n_estimators
Количество деревьев в ансамбле
100 – 1000
learning_rate
Темп обучения
0.001 – 1
max_depth
Максимальная глубина деревьев
3 – 10 (шаг 1)
subsample
Доля данных, используемая при обучении каждого дерева (для борьбы с переобучением)
0.5 – 1.0 (шаг 0.1)
colsample_bytree
Доля признаков, используемых для построения каждого дерева
0.5 – 1.0 (шаг 0.1)

## 🐈 CatBoost (CatBoostClassifier, CatBoostRegressor)

Параметр
Описание
Диапазон перебора
iterations
Количество деревьев
500 – 5000
learning_rate
Темп обучения
0.001 – 1
depth
Глубина деревьев
4 – 10 (шаг 1)
l2_leaf_reg
Коэффициент L2-регуляризации (защита от переобучения)
3 – 10 (шаг 1)
bagging_temperature
Контроль случайности при выборке объектов (чем выше, тем случайнее выборка)
0 – 1 (шаг 0.2)

## 📊 Sklearn (GradientBoostingClassifier, GradientBoostingRegressor)

Параметр
Описание
Диапазон перебора
n_estimators
Количество деревьев
100 – 1000
learning_rate
Темп обучения
0.001 – 1
max_depth
Максимальная глубина деревьев
3 – 10 (шаг 1)
min_samples_split
Минимальное число образцов для разбиения узла
2 – 10 (шаг 2)
subsample
Доля выборки данных для каждого дерева
0.5 – 1.0 (шаг 0.1)

Следует также учесть, что:

Для **learning_rate** сетка
перебора должна быть логарифмической, т.е. перебирать порядковые
значения (к примеру, [1e-3, 1e-2, 1e-1, 1]). В большинстве случаев
достаточно перебрать значения от 1e-5 до 1.

**max_depth** -- максимальная
глубина деревьев в ансамбле. Вообще говоря, эта величина зависит от
числа признаков, но обычно лучше растить небольшие деревья. К примеру,
библиотека CatBoost рекомендует перебирать значения до 10 (и уточняется,
что обычно оптимальная глубина лежит от 6 до 10).

**n_estimators** -- количество
деревьев в ансамбле. Обычно стоит перебирать с каким-то крупным шагом
(можно по логарифмической сетке). Здесь важно найти баланс между
производительностью, временем обучения и качеством. Обычно нескольких
тысяч деревьев бывает достаточно.

**Pro tip:**
Учтите, что в реальных задачах необходимо следить за тем, что
оптимальные значения параметров не попадают на границы интервалов, т.е.
что вы нашли хотя бы локальный минимум. Если Вы перебрали значения
параметра от 1 до 10 и оказалось, что 10 - оптимальное значение, значит
следует перебрать и бОльшие числа, чтобы убедиться, что качество не
улучшается дальше (или по крайней мере убедиться, что рост качества
сильно замедляется и на сильное улучшения рассчитывать не стоит.

### **Задание 1 [2 баллa]**

Данные : датасет со стоимостью поддержанных автомобилей
Метрика : MAE
Цели : В данном задании следует выполнить следующие пункты:

1. Взять две любые библиотеки градиентного бустинга (можете взять самые быстрые)
2. Составить сетку перебора параметров , включающую параметры из таблиц выше.
3. Осуществите перебор параметров по вашей сетке при помощи GridSearchCV на датасетах B и C .
4. Замерьте время перебора.
5. Посчитайте качество модели обученной с оптимальными (с позиции кросс-валидации) параметрами на тренировочном и тестовом множествах.
6. Сделайте выводы о полезности перебора параметров.

**Обратите внимание!**

1. Для всех библиотек вы можете воспользоваться классом GridSearchCV , реализованном в sklearn ,
осуществляющего кросс-валидацию всех параметров и поиска модели с
лучшим качеством. Обратите внимание, что этот класс позволяет установить
количество разбиений датасета, что достаточно сильно влияет на время
работы. Также вы можете воспользоваться n_jobs=-1 для распараллеливания процесса перебора.

**Обратите внимание!**

Если Вы устанавливаете n_jobs для GridSearchCV , то не надо использовать этот параметр для обучаемых регрессоров! Результат может быть плачевным в плане времени...

1. В catboost существует своя реализация перебора параметров, которым можно также воспользоваться (речь о grid_search ).

**Обратите внимание!**

По какой-то причине, которую мне не удалось выяснить, иногда catboost в google colab
работает очень медленно при переборе параметров. Если в соответствующем
задании время обучения catboost занимает не многим более 10 секунд, то в
случае кросс-валидации оно возрастает до 5 минут (соответственно, 5
минут в каждом из узлов). Поэтому используйте catboost на свой страх и
риск :)

In [ ]:
test_parametersXG = {'n_estimators': np.linspace(100, 200, 5).astype('int'),
                     'max_depth': np.arange(1, 10, 2),
                     'learning_rate' : np.logspace(-3, 0, 5)}

test_parametersCB = {'n_estimators': np.linspace(100, 200, 5).astype('int'),
                     'max_depth': np.arange(1, 10, 2),
                     'learning_rate' : np.logspace(-3, 0, 5)}

test_parametersLB = {'n_estimators': np.linspace(100, 200, 5).astype('int'),
                     'max_depth': np.arange(1, 10, 2),
                     'learning_rate' : np.logspace(-3, 0, 5)}

test_parametersSK = {'n_estimators': np.linspace(100, 200, 5).astype('int'),
                     'max_depth': np.arange(1, 10, 2),
                     'learning_rate' : np.logspace(-3, 0, 5)}

In [ ]:
import time
import xgboost
from IPython.display import clear_output

grid_searchXGB_B = GridSearchCV(
    estimator=xgboost.XGBRegressor(),
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersXG,
    n_jobs=-1,
    cv = 3,
    verbose=3
)

grid_searchXGB_C = GridSearchCV(
    estimator=xgboost.XGBRegressor(),
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersXG,
    n_jobs=-1,
    cv = 3,
    verbose=3
)


start_time = time.time()
grid_searchXGB_B.fit(datasets['B'][0], datasets['B'][2])
search_timeB = time.time() - start_time

start_time = time.time()
grid_searchXGB_C.fit(datasets['C'][0], datasets['C'][2])
search_timeC = time.time() - start_time

train_mae_B = mean_absolute_error(datasets['B'][2], grid_searchXGB_B.predict(datasets['B'][0]))
test_mae_B = mean_absolute_error(datasets['B'][3], grid_searchXGB_B.predict(datasets['B'][1]))

train_mae_C = mean_absolute_error(datasets['C'][2], grid_searchXGB_C.predict(datasets['C'][0]))
test_mae_C = mean_absolute_error(datasets['C'][3], grid_searchXGB_C.predict(datasets['C'][1]))

clear_output()

In [ ]:
XGB_B_base = xgboost.XGBRegressor(**test_parameters)
XGB_B_base.fit(datasets['B'][0], datasets['B'][2])
train_mae_B_base = mean_absolute_error(datasets['B'][2], XGB_B_base.predict(datasets['B'][0]))
test_mae_B_base = mean_absolute_error(datasets['B'][3], XGB_B_base.predict(datasets['B'][1]))

XGB_C_base = xgboost.XGBRegressor(**test_parameters)
XGB_C_base.fit(datasets['C'][0], datasets['C'][2])
train_mae_C_base = mean_absolute_error(datasets['C'][2], XGB_C_base.predict(datasets['C'][0]))
test_mae_C_base = mean_absolute_error(datasets['C'][3], XGB_C_base.predict(datasets['C'][1]))



print(f'train base MAE on dataset B: {test_mae_B_base}')
print(f'train base MAE on dataset C: {test_mae_C_base}')

train base MAE on dataset B: 123038.86217726323
train base MAE on dataset C: 18395.56262582478


In [ ]:
df_optimized = pd.DataFrame(columns=['Library', 'Dataset', 'Training time', 'Train MAE', 'Test MAE', 'Default params'])
df_optimized.loc[len(df_optimized)] = {'Library': 'xgboost', 'Dataset': 'B', 'Training time': search_timeB, 'Train MAE': train_mae_B, 'Test MAE': test_mae_B, 'Default params': test_mae_B_base}
df_optimized.loc[len(df_optimized)] = {'Library': 'xgboost', 'Dataset': 'C', 'Training time': search_timeC, 'Train MAE': train_mae_C, 'Test MAE': test_mae_C, 'Default params': test_mae_C_base}


df_optimized

,Library,Dataset,Training time,Train MAE,Test MAE,Default params
0,xgboost,B,108.463798,101777.087323,120672.244346,123038.862177
1,xgboost,C,170.614763,5252.071726,17152.874092,18395.562626


In [ ]:
import lightgbm


grid_searchLGB_B = GridSearchCV(
    estimator=lightgbm.LGBMRegressor(),
    cv=3,
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersLB,
    n_jobs=-1
)

grid_searchLGB_C = GridSearchCV(
    estimator=lightgbm.LGBMRegressor(),
    cv=3,
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersLB,
    n_jobs=-1
)

start_time = time.time()
grid_searchLGB_B.fit(datasets['B'][0], datasets['B'][2])
search_timeB = time.time() - start_time

start_time = time.time()
grid_searchLGB_C.fit(datasets['C'][0], datasets['C'][2])
search_timeC = time.time() - start_time

train_mae_B = mean_absolute_error(datasets['B'][2], grid_searchLGB_B.predict(datasets['B'][0]))
test_mae_B = mean_absolute_error(datasets['B'][3], grid_searchLGB_B.predict(datasets['B'][1]))

train_mae_C = mean_absolute_error(datasets['C'][2], grid_searchLGB_C.predict(datasets['C'][0]))
test_mae_C = mean_absolute_error(datasets['C'][3], grid_searchLGB_C.predict(datasets['C'][1]))

In [ ]:
LGB_B_base = lightgbm.LGBMRegressor(**test_parameters)
LGB_B_base.fit(datasets['B'][0], datasets['B'][2])
train_mae_B_base = mean_absolute_error(datasets['B'][2], LGB_B_base.predict(datasets['B'][0]))
test_mae_B_base = mean_absolute_error(datasets['B'][3], LGB_B_base.predict(datasets['B'][1]))

LGB_C_base = lightgbm.LGBMRegressor(**test_parameters)
LGB_C_base.fit(datasets['C'][0], datasets['C'][2])
train_mae_C_base = mean_absolute_error(datasets['C'][2], LGB_C_base.predict(datasets['C'][0]))
test_mae_C_base = mean_absolute_error(datasets['C'][3], LGB_C_base.predict(datasets['C'][1]))

In [ ]:
df_optimized2 = pd.DataFrame(columns=['Library', 'Dataset', 'Training time', 'Train MAE', 'Test MAE', 'Default params'])
df_optimized2.loc[len(df_optimized2)] = {'Library': 'LightGBM', 'Dataset': 'B', 'Training time': search_timeB, 'Train MAE': train_mae_B, 'Test MAE': test_mae_B, 'Default params': test_mae_B_base}
df_optimized2.loc[len(df_optimized2)] = {'Library': 'LightGBM', 'Dataset': 'C', 'Training time': search_timeC, 'Train MAE': train_mae_C, 'Test MAE': test_mae_C, 'Default params': test_mae_C_base}


df_optimized2

,Library,Dataset,Training time,Train MAE,Test MAE,Default params
0,LightGBM,B,151.754793,101777.087323,120672.244346,122343.854494
1,LightGBM,C,175.081361,5252.071726,17152.874092,31055.237371


In [ ]:
df_allres = pd.concat([df_optimized, df_optimized2], axis = 0)
df_allres

,Library,Dataset,Training time,Train MAE,Test MAE,Default params
0,xgboost,B,109.633170,101777.087323,120672.244346,123038.862177
1,xgboost,C,179.483378,5252.071726,17152.874092,18395.562626
0,LightGBM,B,58.245621,101563.515817,122031.833740,122343.854494
1,LightGBM,C,55.553446,19396.921568,28645.448950,31055.237371


Ваши выводы: Перебор гиперпараметров позволил значительно улучшить качество моделей обеих библиотек и уменьшить МАЕ во всех четырёх случаях.

Аналогично первому ноутбуку по градиентому бустингу датасет С показывает лучшие резульаты

### **Задание 1 [2 балла]**

Данные : датасет со стоимостью поддержанных автомобилей
Метрика : MAE
Цели : В данном задании следует выполнить следующие пункты:

1. Выполнить задание 1 с использованием всех библиотек
(для каждой библиотеки можно использовать свою сетку перебора).
Разрешается использовать маленькие сетки с небольшим числом узлов, но не
менее 10.
2. Вывести ниже результаты работы с нашими дефолтными test_parameters параметрами, а также с оптимальными.
3. Вы будете получать **0.5 баллa** за
каждую библиотеку с использованием которой точность ваших оптимальных
параметров превзойдет качество наших дефолтных параметров. Таким
образом, максимальный балл за задание равен числу библиотек, **2 балла.**

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

grid_searchSGB_B = GridSearchCV(
    estimator=GradientBoostingRegressor(),
    cv=3,
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersSK,
    n_jobs=-1
)

grid_searchSGB_C = GridSearchCV(
    estimator=GradientBoostingRegressor(),
    cv=3,
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersSK,
    n_jobs=-1
)

start_time = time.time()
grid_searchSGB_B.fit(datasets['B'][0], datasets['B'][2])
search_timeB = time.time() - start_time

start_time = time.time()
grid_searchSGB_C.fit(datasets['C'][0], datasets['C'][2])
search_timeC = time.time() - start_time

train_mae_B = mean_absolute_error(datasets['B'][2], grid_searchSGB_B.predict(datasets['B'][0]))
test_mae_B = mean_absolute_error(datasets['B'][3], grid_searchSGB_B.predict(datasets['B'][1]))

train_mae_C = mean_absolute_error(datasets['C'][2], grid_searchSGB_C.predict(datasets['C'][0]))
test_mae_C = mean_absolute_error(datasets['C'][3], grid_searchSGB_C.predict(datasets['C'][1]))

In [ ]:
SGB_B_base = GradientBoostingRegressor(**test_parameters)
SGB_B_base.fit(datasets['B'][0], datasets['B'][2])
train_mae_B_base = mean_absolute_error(datasets['B'][2], SGB_B_base.predict(datasets['B'][0]))
test_mae_B_base = mean_absolute_error(datasets['B'][3], SGB_B_base.predict(datasets['B'][1]))

SGB_C_base = GradientBoostingRegressor(**test_parameters)
SGB_C_base.fit(datasets['C'][0], datasets['C'][2])
train_mae_C_base = mean_absolute_error(datasets['C'][2], SGB_C_base.predict(datasets['C'][0]))
test_mae_C_base = mean_absolute_error(datasets['C'][3], SGB_C_base.predict(datasets['C'][1]))

In [ ]:
df_optimized3 = pd.DataFrame(columns=['Library', 'Dataset', 'Training time', 'Train MAE', 'Test MAE', 'Default params'])
df_optimized3.loc[len(df_optimized3)] = {'Library': 'sklearn', 'Dataset': 'B', 'Training time': search_timeB, 'Train MAE': train_mae_B, 'Test MAE': test_mae_B, 'Default params': test_mae_B_base}
df_optimized3.loc[len(df_optimized3)] = {'Library': 'sklearn', 'Dataset': 'C', 'Training time': search_timeC, 'Train MAE': train_mae_C, 'Test MAE': test_mae_C, 'Default params': test_mae_C_base}


df_optimized3

,Library,Dataset,Training time,Train MAE,Test MAE,Default params
0,sklearn,B,385.570789,98846.403618,121795.759913,123291.984067
1,sklearn,C,645.276778,4625.280200,17138.314744,18109.423838


In [ ]:
import catboost


grid_searchCB_B = GridSearchCV(
    estimator=CatBoostRegressor(),
    cv=3,
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersCB,
    n_jobs=-1
)

grid_searchCB_C = GridSearchCV(
    estimator=CatBoostRegressor(),
    cv=3,
    scoring='neg_mean_absolute_error',
    param_grid=test_parametersCB,
    n_jobs=-1
)

start_time = time.time()
grid_searchCB_B.fit(datasets['B'][0], datasets['B'][2])
search_timeB = time.time() - start_time

start_time = time.time()
grid_searchCB_C.fit(datasets['C'][0], datasets['C'][2])
search_timeC = time.time() - start_time

train_mae_B = mean_absolute_error(datasets['B'][2], grid_searchCB_B.predict(datasets['B'][0]))
test_mae_B = mean_absolute_error(datasets['B'][3], grid_searchCB_B.predict(datasets['B'][1]))

train_mae_C = mean_absolute_error(datasets['C'][2], grid_searchCB_C.predict(datasets['C'][0]))
test_mae_C = mean_absolute_error(datasets['C'][3], grid_searchCB_C.predict(datasets['C'][1]))

In [ ]:
CB_B_base = GradientBoostingRegressor(**test_parameters)
CB_B_base.fit(datasets['B'][0], datasets['B'][2])
train_mae_B_base = mean_absolute_error(datasets['B'][2], CB_B_base.predict(datasets['B'][0]))
test_mae_B_base = mean_absolute_error(datasets['B'][3], CB_B_base.predict(datasets['B'][1]))

CB_C_base = GradientBoostingRegressor(**test_parameters)
CB_C_base.fit(datasets['C'][0], datasets['C'][2])
train_mae_C_base = mean_absolute_error(datasets['C'][2], CB_C_base.predict(datasets['C'][0]))
test_mae_C_base = mean_absolute_error(datasets['C'][3], CB_C_base.predict(datasets['C'][1]))

In [ ]:
df_optimized4 = pd.DataFrame(columns=['Library', 'Dataset', 'Training time', 'Train MAE', 'Test MAE', 'Default params'])
df_optimized4.loc[len(df_optimized4)] = {'Library': 'catboost', 'Dataset': 'B', 'Training time': search_timeB, 'Train MAE': train_mae_B, 'Test MAE': test_mae_B, 'Default params': test_mae_B_base}
df_optimized4.loc[len(df_optimized4)] = {'Library': 'catboost', 'Dataset': 'C', 'Training time': search_timeC, 'Train MAE': train_mae_C, 'Test MAE': test_mae_C, 'Default params': test_mae_C_base}


df_optimized4

,Library,Dataset,Training time,Train MAE,Test MAE,Default params
0,catboost,B,351.909354,113776.225728,120409.511806,123355.183934
1,catboost,C,374.070330,10099.227713,17936.239125,18182.806174


In [ ]:
df_allres = pd.concat([df_optimized, df_optimized2, df_optimized3, df_optimized4], axis = 0)
df_allres

,Library,Dataset,Training time,Train MAE,Test MAE,Default params
0,xgboost,B,108.463798,101777.087323,120672.244346,123038.862177
1,xgboost,C,170.614763,5252.071726,17152.874092,18395.562626
0,LightGBM,B,151.754793,101777.087323,120672.244346,122343.854494
1,LightGBM,C,175.081361,5252.071726,17152.874092,31055.237371
0,sklearn,B,385.570789,98846.403618,121795.759913,123291.984067
1,sklearn,C,645.276778,4625.280200,17138.314744,18109.423838
0,catboost,B,351.909354,113776.225728,120409.511806,123355.183934
1,catboost,C,374.070330,10099.227713,17936.239125,18182.806174


**Ваши пояснения для проверяющих (опционально):** Добавила Catboost и Sklearn

Обычнно перебор параметров и поиск по сетке это самая скучная часть
работы, поскольку занимает много времени, но не гарантирует
воспроизведение результата при небольшом изменении датасета, да и сетку
надо переосмысливать при каждом обновлении.

Но сейчас мы поймём, что этого можно избежать, поскольку есть библиотека, которая всё сделает за нас!

Эмоции выполняющего в этот момент.png

Нашего спасителя зовут HyperOpt . На первый взгляд hyperopt делает всё то же самое, что и grid search ,
а именно перебирает параметры. По факту же hyperopt превращает это в
задачу оптимизации, используя некоторые эвристики для ускорения
сходимости процесса. К тому же, он требует лишь информацию о границе
интервалов, а не сами сетки. В теории это должно помочь нам добиться
лучших результатов за более короткое время. Давайте попробуем это
сделать.

Для данного эксперимента я рекомендую использовать lightgbm , поскольку она быстрее и с ней удобнее играться, но Вы можете воспользоваться любой библиотекой из представленных выше.

### **Задание 2 [Bonus][3 балла]**

Данные : датасет со стоимостью поддержанных автомобилей

Метрика : MAE

Цели : В данном задании следует выполнить следующие пункты:

1. Взять любую библиотеку градиентного бустинга (можете взять самую быструю)
2. Составить сетку перебора в hyperopt , включающую параметры n_estimators , max_depth и learning_rate в hyperopt. Вам могут понадобиться такие типы данных, как hp.choise , hp.qloguniform , hp.uniform и hp.quniform (можно также пользоваться np.arange ). Также для округления значения типа float до целых чисел (4.0 -> 4) используйте scope.int .
3. Реализуйте функцию, которая принимает на вход словарь параметров для регрессора, и при помощи cv оценивает его качество на датасете A
(можно воспользоваться cross_val_score, а для ускорения поставить
cv=3). Не забудьте о том, в каком виде lightgbm принимает категориальные
признаки в numpy и что также надо передавать индексы категориальных
признаков.
4. Создайте объект trials=Trials() , который будет хранить информацию о процессе оптимизации.
5. Используя функцию fmin , оптимизируйте Вашу функцию. Установите algo=tpe.suggest, trials=trials и max_evals , по крайней мере, 50. verbose=1 позволит видеть прогресс-бар по типу tqdm.
6. Выведите получившиеся параметры. Нарисуйте график , показывающий значение loss в ходе оптимизации. Посчитайте качество на тесте при использовании лучших параметров (возвращаются после использования fmin). Сделайте выводы по результату.

In [ ]:
import lightgbm
from hyperopt import STATUS_OK

trials=Trials()
categorical_features = dfA.select_dtypes(include=['category']).columns.tolist()
cat_feature_indices = [datasets['A'][0].columns.get_loc(col) for col in categorical_features]


space = {
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.001), np.log(0.5)),
}

def objective(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])

    LGB_B_base = lightgbm.LGBMRegressor(**params)

    score = cross_val_score(LGB_B_base, datasets['A'][0], datasets['A'][2], cv=3, scoring='neg_mean_absolute_error', fit_params={'categorical_feature': cat_feature_indices})

    return {'loss': -score.mean(), 'status': STATUS_OK}


best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    verbose=1
)

Выходные данные были обрезаны до нескольких последних строк (5000).
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

losses = [trial['result']['loss'] for trial in trials.trials]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(losses, 'bo-')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss per iteration')

plt.subplot(1, 2, 2)
plt.plot(np.minimum.accumulate(losses), 'ro-')
plt.xlabel('Iteration')
plt.ylabel('Best Loss')
plt.title('Best loss progression')

plt.tight_layout()
plt.show()

Ваши выводы:

## Предсказание зрительских симпатий

Ну что, детишки, а теперь перейдём к действительно важным вопросам.

**Обратите внимание!** Следующее задание сдается в системе cv-gml.ru .
Для выполнения этого задания необходимо скачать датасет из задания.
Здесь вы можете немного почитать про датасет и, при желании,
поэкспериментировать. На cv-gml.ru загружайте уже готовый скрипт с подобранными параметрами для обучаемого регрессора. Релизовать код необходимо в шаблонном файле awards_prediction.py , который вы можете найти в проверяющей системе.

В некотором царстве, некотором государстве была развита
кинопромышленность. Новые фильмы в этом государстве показывают по
интернету, а пользователи после просмотра могут дать фильму некоторую
"награду". Наша цель - предсказать число наград для фильма.

В нашем распоряжении имеются следующие данные:

awards - количество наград, полученных фильмом от пользователей (целевое значение)
potions - количество магических зелий, потраченных на создание спец-эффектов
genres - жанры созданного фильма
questions - количество вопросов, заданных пользователями на соответствующих форумах об этом фильме до премьеры
directors - режиссеры фильма (если неизвестны, то unknown)
filming_locations - области, в которых снимался фильм
runtime - продолжительность фильма в некоторых единицах, принятых в этом государстве
critics_liked - количество критиков из 100, присудивших награды фильму на предварительных закрытых показах
pre-orders - количество зрителей, заранее купивших билеты на первый показ
keywords - ключевые слова, описывающие содержание фильма
release_year - год, во котором фильм был показан (конечно, в летоисчислении этого государства)

Следующие поля появляются несколько раз с разными значениями i:

actor_i_known_movies - количество известных фильмов актера i (i от 0 до 2)

actor_i_postogramm - количество подписчиков в социальной сети "по сто грамм" актера i (i от 0 до 2)

actor_i_gender - пол актера i (i от 0 до 2)

actor_i_age - возраст актера i (i от 0 до 2)

**Обратите внимание!** Учтите, что при
OHE кодировании признаки на обучении и тестировании должны совпадать!
Если вы примените простое .get_dummies() или что-то подобное, то
признаки на трейне и тесте получатся разные! Так что вам, вероятно,
придётся придумать способ для того, чтобы сохранить их :)

**Подсказка** для работы с текстом можно воспользоваться методом TF-IDF (ключевые слова: TfIdfTransformer ). Также может быть полезен CountVectorizer . Только учтите, что никто не гарантирует улучшение результата с использованием данных методов ;)

**Обратите внимание!** В проверяющей
системе имеется проблема с catboost. При использовании этой библиотеки, в
скрипте с решением необходимо инициализировать метод с использованием train_dir как показано тут:
CatBoostRegressor(train_dir='/tmp/catboost_info')

### **Задание 3 [6 баллов, ML-решение, не проверяется на кросс-проверке]**

Данные : датасет с ценами поддержанных автомобилей
Метрика : MAE
Цели : В данном задании следует выполнить следующие пункты:

1. Взять любую библиотеку градиентного бустинга
2. Используя предложенный датасет, обучить регрессор для предсказания
awards (предоставляем полную свободу в настройках и выборе методов)
3. Загрузить решение и получить качество на закрытой выборке больше порогового значения

In [ ]:
## your efficient code here

## Конец

Ну что детишки... Можете добавлять еще 4 библиотеки в своё резюме датасаентиста!

```
(╯°□°)╯︵ ┻━┻ FLIP THAT TABLE.

┻━┻ ︵ ヽ(°□°ヽ) FLIP THIS TABLE.

┻━┻ ︵ ＼\('0')/／ ︵ ┻━┻ FLIP ALL THE TABLES

ಠ_ಠ Son... ಠ_ಠ Put. ಠ__ಠ The tables. ಠ___ಠ Back.

(╮°-°)╮┳━┳

(╯°□°)╯︵ ┻━┻ NEVER!!!!
```